# artifacts: DINOv2 + CatBoost

Используем готовые признаки шести рендеров. Проверяем, помогают ли изображения отдельно и вместе с геометрией.

## Что хотим проверить

Берём готовые признаки шести ракурсов. DINOv2 не обучаем, обучается только классификатор поверх них. Сравниваем объединение ракурсов и добавление геометрии.

$$z=[mean(z_1,\ldots,z_6),\ max(z_1,\ldots,z_6)].$$

Среднее передаёт общий вид, максимум сохраняет сильный сигнал хотя бы с одного ракурса. Например, дефект может быть виден только сзади. Также в опытах встречается конкатенация всех шести векторов.

## 1. Данные и прежнее разбиение

In [ ]:
%pip install -q catboost iterative-stratification

In [ ]:
from pathlib import Path
import hashlib
import json
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from catboost import CatBoostClassifier
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data/train.csv').exists()), None)
if ROOT is None:
    ROOT = Path('/content/drive/MyDrive/Colab Notebooks/colab_upload/AI challenge main')
import sys
sys.path.insert(0, str(ROOT))
from src.evaluation import binary_metrics, best_threshold
from src.data import split_indices, load_embeddings
from src.plots import setup_plot_style
setup_plot_style()
package = ROOT / 'noisy_dinov2_assets.zip'
if package.is_file():
    with zipfile.ZipFile(package) as archive:
        for name in archive.namelist():
            if not (ROOT / name).is_file():
                archive.extract(name, ROOT)
assert (ROOT / 'data/train.csv').is_file(), ROOT

labels = pd.read_csv(ROOT / 'data/train.csv')
targets = labels.columns.drop('item_id').tolist()
train_idx, val_idx, test_idx = split_indices(labels)
train = labels.iloc[train_idx].reset_index(drop=True)
val = labels.iloc[val_idx].reset_index(drop=True)
objects = pd.concat([train, val], ignore_index=True)
assert (len(train), len(val), len(test_idx)) == (6259, 1361, 1344)
assert set(train.item_id).isdisjoint(val.item_id)
print('Train:', len(train), '| Validation:', len(val), '| artifacts:', int(val.artifacts.sum()))

## 2. Готовые DINOv2 признаки

У каждого объекта шесть embeddings по 384 числа. Берём среднее и максимум по ракурсам.

In [ ]:
cache = ROOT / 'labels/simple/models/dinov2'
embedding_file = cache / 'embeddings.npy'
if not embedding_file.is_file():
    cache = ROOT / 'labels/noisy/artifacts'
    embedding_file = cache / 'dinov2_embeddings.npy'
assert embedding_file.is_file(), embedding_file
expected = {
    'model': 'dinov2_vits14',
    'revision': '7764ea0f912e53c92e82eb78a2a1631e92725fc8',
    'image_size': 224, 'interpolation': 'bicubic',
    'mean': [.485, .456, .406], 'std': [.229, .224, .225],
}
views = load_embeddings(embedding_file, cache / 'embeddings_manifest.json',
                        cache / 'embeddings_done.npy', objects.item_id,
                        expected, ROOT / 'data/train.csv')
visual = np.concatenate([views.mean(axis=1), views.max(axis=1)], axis=1)
print('DINOv2:', visual.shape)

## 3. Сохранённая геометрия

Берём 44 базовых признака, выбранные признаки `lowpoly` и `noisy`, а также локальные рёбра. Компоненты пока не добавляем: они снизили AP в предыдущем опыте.

In [ ]:
config = json.loads((ROOT / 'archive/artifacts/catboost_features_config.json').read_text(encoding='utf-8'))
base = pd.read_csv(ROOT / 'archive/artifacts/train_mesh_features_relative.csv').set_index('item_id')
lowpoly = pd.read_csv(ROOT / 'labels/lowpoly/artifacts/geometry_features.csv').set_index('item_id')
noisy = pd.read_csv(ROOT / 'labels/noisy/artifacts/smoothing_features.csv').set_index('item_id')
local = pd.read_csv(ROOT / 'labels/artifacts/artifacts/local_edge_features.csv').set_index('item_id')
lowpoly_columns = ['triangle_aspect_p90', 'triangle_near_degenerate_share',
                   'dihedral_over_30_share', 'pca_curvature_p95',
                   'normal_variation_p95', 'boundary_edge_share',
                   'nonmanifold_edge_share', 'duplicate_vertex_share',
                   'invalid_faces', 'degenerate_faces']
noisy_columns = [name for name in noisy if name.startswith('smooth_')]
local_columns = ['local_edge_p95', 'local_edge_p99', 'local_edge_over_2', 'local_edge_over_4']
geometry = (base[config['feature_columns']]
            .join(lowpoly[lowpoly_columns])
            .join(noisy[noisy_columns])
            .join(local[local_columns])
            .loc[objects.item_id])
assert geometry.shape == (7620, 72)
assert not np.isinf(geometry.to_numpy(dtype=float)).any()
print('Геометрия:', geometry.shape)

## 4. Два CatBoost и сравнение

In [ ]:
out = ROOT / 'labels/artifacts/models/dinov2'
out.mkdir(parents=True, exist_ok=True)
y_train = train.artifacts.to_numpy()
y_val = val.artifacts.to_numpy()
n_train = len(train)
grid = np.arange(.01, 1, .01)

def score(probability, threshold):
    result = binary_metrics(y_val, probability, threshold)
    return {key: result[key] for key in ['Precision', 'Recall', 'F1', 'AP', 'FP', 'FN', 'Threshold']}

results = []
probabilities = val[['item_id', 'artifacts']].copy()
for name, X in {'DINOv2': visual,
                'DINOv2_geometry': np.concatenate([visual, geometry.to_numpy()], axis=1)}.items():
    model = CatBoostClassifier(**config['parameters'])
    model.fit(X[:n_train], y_train, eval_set=(X[n_train:], y_val),
              early_stopping_rounds=50, verbose=False)
    p = model.predict_proba(X[n_train:])[:, 1]
    threshold = best_threshold(y_val, p, grid)
    results.append({'Model': name, **score(p, threshold)})
    probabilities[name] = p
    model.save_model(str(out / f'{name}.cbm'))
    print(name, '| F1:', round(results[-1]['F1'], 4), '| AP:', round(results[-1]['AP'], 4), flush=True)

table = pd.DataFrame(results)
display(table.round(4))
table.to_csv(out / 'metrics.csv', index=False)
probabilities.to_csv(out / 'validation_probabilities.csv', index=False)
display(pd.concat([
    pd.DataFrame([{'Model': 'Old ResNet18 + CatBoost', 'F1': .3153},
                  {'Model': 'Mesh CatBoost + local edges', 'F1': .2133}]),
    table[['Model', 'F1']]
], ignore_index=True).round(4))

## 5. Ошибки лучшего варианта

In [ ]:
best_row = table.loc[table.F1.idxmax()]
best_name, threshold = best_row.Model, best_row.Threshold
cases = val[['item_id', 'artifacts', 'noisy', 'lowpoly', 'set']].copy()
cases['probability'] = probabilities[best_name].to_numpy()
cases['prediction'] = (cases.probability >= threshold).astype(int)

def show_cases(rows):
    fig, axes = plt.subplots(1, len(rows), figsize=(25, 4))
    for ax, (_, row) in zip(axes, rows.iterrows()):
        with Image.open(ROOT / 'data/train' / f'{row.item_id}.png') as image:
            ax.imshow(image)
        ax.set_title(f'{row.item_id[:8]} | p={row.probability:.2f}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()
    display(rows.round(3))

print('Ложные срабатывания:')
show_cases(cases[(cases.artifacts == 0) & (cases.prediction == 1)].nlargest(5, 'probability'))
print('Пропуски:')
show_cases(cases[(cases.artifacts == 1) & (cases.prediction == 0)].nsmallest(5, 'probability'))

## Вывод

DINOv2 mean + max дал F1 = 0.3671 и AP = 0.2603. Добавление геометрии дало F1 = 0.3717 и AP = 0.2724. Оба варианта лучше старой связки ResNet18 + CatBoost (F1 = 0.3153), но прирост от геометрии небольшой. У варианта с геометрией 119 FP и 23 FN; 79 FP имеют метку `noisy`. На просмотренных FN есть небольшие повреждения отдельных участков, которые глобальные embeddings могут пропускать. Следующий опыт — локальные patch-признаки и MIL. Порог и модель выбирались на одной validation, поэтому F1 может быть оптимистичным; internal test не использовали.

## Вывод

Сравниваем на прежней validation. Test не используем для выбора порогов. Небольшой прирост после многих опытов ещё нужно перепроверять.